# SENTINEL — YOLOv8s Weapon Training (Phase 3)

Asal training notebook: Drive se dataset → verification → training → evaluation → weights Drive per.

**Tareeqa:** Sirf Cell 1 (CONFIG) mai `RUN_MODE` set karo — pehle `"smoke"` (10 epochs, pipeline check), phir Muse se confirm karke `"baseline"` (50 epochs). Baqi cells tartib se chalao (Shift+Enter). Kisi cell mai red error aye to **wahin ruk jao** aur output Muse ko bhejo, khud fix mat karo.

Training script repo se ata hai (`scripts/training/train_weapon_detector.py`) — notebook use clone karke chalati hai, taake Colab aur repo hamesha ek hi code use karein.

In [ ]:
# ============ CONFIG — sirf yahi cell badalna hai ============
RUN_MODE = "smoke"        # "smoke" = 10 epochs (pipeline check) | "baseline" = 50 epochs (asal run)
REPO_BRANCH = "feature/weapon-training"   # branch merge ho jane ke baad "main" kar dena

from datetime import datetime
from pathlib import Path

EPOCHS = 10 if RUN_MODE == "smoke" else 50
RUN_NAME = f"weapon_yolov8s_{RUN_MODE}_{datetime.now():%Y%m%d_%H%M}"

DATASET_ZIP = Path("/content/drive/MyDrive/SENTINEL_FYP/datasets/weapon_yolo.zip")
DRIVE_RUNS = Path("/content/drive/MyDrive/SENTINEL_FYP/runs")
WORK = Path("/content/datasets")
RUNS_PROJECT = "/content/runs/weapon"
MODEL_PATH = "/content/yolov8s.pt"

print("RUN_MODE :", RUN_MODE, "| EPOCHS:", EPOCHS)
print("RUN_NAME :", RUN_NAME)

## Step 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU nahi mila — Runtime > Change runtime type > T4 GPU select karo, phir dobara chalao."

## Step 2 — Drive mount aur dataset zip ki mojoodgi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

assert DATASET_ZIP.exists(), f"Dataset zip nahi mila: {DATASET_ZIP} — Drive shortcut wala step check karo aur Muse ko batao."
print("Dataset zip mila:", DATASET_ZIP.name, "| size (MB):", round(DATASET_ZIP.stat().st_size / 1e6, 1))

## Step 3 — Ultralytics install + model weights ki mazboot download
GitHub 504 ka sabak: `wget -c` resume ke saath download hogi, fail ho to yehi cell dobara chalana — download wahi se resume hogi.

In [ ]:
%pip install -q ultralytics
!wget -c --tries=10 --waitretry=5 "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8s.pt" -O /content/yolov8s.pt

size = Path(MODEL_PATH).stat().st_size
print("yolov8s.pt size (MB):", round(size / 1e6, 2))
assert size > 20_000_000, "Weights file adhoori lagti hai — ye cell dobara chalao (resume hogi)."

## Step 4 — Repo clone (training script wahi se ayega)

In [ ]:
!rm -rf /content/sentinel_repo
!git clone -b $REPO_BRANCH --single-branch https://github.com/muddassir1935/FYP.git /content/sentinel_repo

script = Path("/content/sentinel_repo/scripts/training/train_weapon_detector.py")
assert script.exists(), "Training script nahi mila — branch ka naam CONFIG mai check karo, Muse ko batao."
print("Training script ready:", script)

## Step 5 — Dataset unzip + verification (counts match na hon to training yahi ruk jayegi)

In [ ]:
import zipfile

WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(WORK)

# dataset root = wo folder jiske andar images/ aur labels/ dono hon (zip ki structure dono surton mai handle hai)
cands = [p.parent for p in WORK.rglob("images") if (p.parent / "labels").exists()]
assert cands, "Dataset structure samajh nahi aya — ye output Muse ko bhejo."
root = sorted(cands, key=lambda p: len(p.parts))[0]
print("Dataset root:", root)

IMG_EXTS = {".jpg", ".jpeg", ".png"}
expected = {"train": 4731, "val": 1351, "test": 680}
ok = True
for split, exp in expected.items():
    imgs = [p for p in (root / "images" / split).glob("*") if p.suffix.lower() in IMG_EXTS]
    lbls = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images = {len(imgs)} (expected {exp}), labels = {len(lbls)}")
    if len(imgs) != exp:
        ok = False

# labels sanity: har line ki pehli value (class id) sirf 0 (Gun) ya 1 (Knife) ho
bad, checked = 0, 0
for txt in (root / "labels").rglob("*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            checked += 1
            if int(float(line.split()[0])) not in (0, 1):
                bad += 1
print(f"label lines checked = {checked}, bad class ids = {bad}")

assert ok and bad == 0, "DATASET VERIFICATION FAILED — training roki gayi. Ye poora output Muse ko bhejo."

# Colab ke liye data yaml (repo wali yaml local paths ke liye hai, ye wali /content ke absolute path ke saath)
DATA_YAML = root / "sentinel_data.yaml"
DATA_YAML.write_text(f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 2\nnames: ['Gun', 'Knife']\n")
print("DATA_YAML:", DATA_YAML)
print("DATASET VERIFIED OK")

## Step 6 — Training (smoke ~15–25 min, baseline andazan 1.5–3 ghante — session khula rakho)

In [ ]:
%cd /content/sentinel_repo
!python scripts/training/train_weapon_detector.py --data $DATA_YAML --model $MODEL_PATH --epochs $EPOCHS --imgsz 640 --batch 16 --project $RUNS_PROJECT --name $RUN_NAME

## Step 7 — Results (test split per final numbers + confusion matrix)

In [ ]:
import json
from IPython.display import Image, display

run_dir = Path(RUNS_PROJECT) / RUN_NAME
summary = json.loads((run_dir / "run_summary.json").read_text())
print("TEST METRICS:", json.dumps(summary["test_metrics"], indent=2))

cm = run_dir / "confusion_matrix.png"
if cm.exists():
    display(Image(str(cm), width=640))

## Step 8 — Weights Drive per save + run record (COPY block Muse ko bhejna hai)

In [ ]:
import hashlib
import shutil

dest = DRIVE_RUNS / RUN_NAME
dest.mkdir(parents=True, exist_ok=True)

best = run_dir / "weights" / "best.pt"
dated = dest / f"{RUN_NAME}_best.pt"
shutil.copy(best, dated)
for extra in ["results.csv", "confusion_matrix.png", "results.png", "run_summary.json"]:
    src = run_dir / extra
    if src.exists():
        shutil.copy(src, dest / extra)

sha = hashlib.sha256(dated.read_bytes()).hexdigest()
tm = summary["test_metrics"]
print("=" * 60)
print("===== COPY FROM HERE — ye block Muse ko bhej dena =====")
print(f"RUN OK — {RUN_NAME}")
print(f"Weights file : {dated.name}")
print(f"Size (MB)    : {dated.stat().st_size / 1e6:.2f}")
print(f"SHA-256      : {sha}")
print(f"Test mAP@0.5 : {tm.get('mAP50')} | mAP@0.5:0.95 : {tm.get('mAP50_95')} | P : {tm.get('precision')} | R : {tm.get('recall')} | F1 : {tm.get('f1')}")
for cls, vals in tm.get("per_class", {}).items():
    print(f"  {cls}: AP50 = {vals.get('AP50')}")
print("===== COPY TO HERE =====")
print("=" * 60)